# Notebook 03 — Data Curation
## Amazon LA Delivery Failure Prediction

**Program:** Correlation One — DANA, Week 12  
**Date:** April 2026

---

This notebook covers the full data curation pipeline:
1. **Data Sourcing** — dataset origin and loading
2. **Data Profiling** — structure, content, and relationship discovery
3. **Data Wrangling** — encoding, cleaning, final schema

In [ ]:
# ── Standard imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['axes.facecolor']   = 'white'

AMAZON_ORANGE = '#FF9900'
AMAZON_NAVY   = '#232F3E'
AMAZON_BLUE   = '#146EB4'

print('Libraries loaded successfully')

---
## Part 1 — Data Sourcing

The dataset is a synthetic recreation of Amazon LA last-mile operations, generated with `data/generate_data.py`. It captures operational patterns across three splits:

In [ ]:
# Load all three data splits
train = pd.read_csv('../data/packages_validation.csv')
val   = pd.read_csv('../data/packages_validation.csv')
test  = pd.read_csv('../data/packages_test.csv')

# Dataset inventory
print('DATA SOURCING — Dataset Inventory')
print('=' * 55)
for name, df in [('packages_validation.csv', train),
                  ('packages_validation.csv', val),
                  ('packages_test.csv', test)]:
    print(f'  {name:<30} → {len(df):,} rows | {df.shape[1]} columns')
    print(f'    Failure rate: {df["delivery_failed"].mean():.1%} | '
          f'Size: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'\n  Total records: {len(train)+len(val)+len(test):,}')

# Use train for profiling
df = train.copy()
print(f'\nPrimary analysis: packages_validation.csv ({len(df):,} records)')

---
## Part 2 — Data Profiling

### 2.1 Structure Discovery — dtypes, ranges, cardinality

In [ ]:
# Structure discovery — comprehensive profile
profile = []
for col in df.columns:
    col_data = df[col]
    if pd.api.types.is_numeric_dtype(col_data):
        profile.append({
            'Column': col,
            'Dtype': str(col_data.dtype),
            'Nulls': col_data.isnull().sum(),
            'Distinct': col_data.nunique(),
            'Min': f'{col_data.min():.2f}' if col_data.dtype == float else col_data.min(),
            'Max': f'{col_data.max():.2f}' if col_data.dtype == float else col_data.max(),
            'Mean/Mode': f'{col_data.mean():.2f}'
        })
    else:
        profile.append({
            'Column': col,
            'Dtype': str(col_data.dtype),
            'Nulls': col_data.isnull().sum(),
            'Distinct': col_data.nunique(),
            'Min': col_data.min(),
            'Max': col_data.max(),
            'Mean/Mode': col_data.mode()[0]
        })

profile_df = pd.DataFrame(profile)
print('Structure Discovery — Column Profile')
print(profile_df.to_string(index=False))

### 2.2 Content Discovery — Missing values, distributions, outliers

In [ ]:
# Missing values heatmap
fig, ax = plt.subplots(figsize=(12, 4))
missing = df.isnull().astype(int)

if missing.sum().sum() == 0:
    ax.text(0.5, 0.5, 'No missing values detected across all 13 columns',
            ha='center', va='center', fontsize=16, color='green',
            fontweight='bold', transform=ax.transAxes)
    ax.set_facecolor('#f8fff8')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)
else:
    sns.heatmap(missing.T, cmap='Reds', cbar=False, ax=ax, yticklabels=df.columns)

ax.set_title('Missing Values Audit — Amazon LA Delivery Dataset',
             fontsize=13, fontweight='bold', color=AMAZON_NAVY, pad=10)
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total null values: {df.isnull().sum().sum()} — Data quality: PASS')

In [ ]:
# Categorical distributions
cat_cols = ['package_type', 'shift', 'carrier', 'weather_risk']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=AMAZON_ORANGE,
                  edgecolor='white', linewidth=0.5)
    ax.set_title(f'{col} Distribution', fontweight='bold', color=AMAZON_NAVY)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val:,}\n({val/len(df):.0%})', ha='center', fontsize=8.5, color=AMAZON_NAVY)

plt.suptitle('Categorical Feature Distributions — Training Set (n=3,559)',
             fontsize=14, fontweight='bold', color=AMAZON_NAVY)
plt.tight_layout()
plt.savefig('../reports/figures/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Binary feature rates with business context
binary_cols = ['double_scan', 'short_service_time', 'delivery_failed', 'cr_number_missing', 'delivery_failed']
binary_labels = ['Double Scan\n(Scan Error)', 'Locker Issue\n(Infra Failure)',
                 'Damaged on\nArrival', 'CR Number\nMissing', 'Delivery\nFailed (TARGET)']
rates = [df[col].mean() * 100 for col in binary_cols]
colors = [AMAZON_BLUE, AMAZON_BLUE, AMAZON_ORANGE, AMAZON_BLUE, '#CC0000']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(binary_labels, rates, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Positive Rate (%)')
ax.set_title('Binary Feature Positive Rates — Operational Flag Analysis',
             fontsize=13, fontweight='bold', color=AMAZON_NAVY, pad=10)
ax.axhline(y=19.4, color='#CC0000', linestyle='--', linewidth=1.5, label='Baseline failure rate (0.70%)', alpha=0.6)
ax.legend(fontsize=9)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{rate:.1f}%', ha='center', fontweight='bold', color=AMAZON_NAVY)

plt.tight_layout()
plt.savefig('../reports/figures/binary_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Numerical feature distributions — outlier detection
num_cols = ['route_distance_km', 'packages_in_route', 'days_in_fc']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, num_cols):
    ax.hist(df[col], bins=30, color=AMAZON_ORANGE, edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(df[col].mean(), color=AMAZON_NAVY, linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color=AMAZON_BLUE, linestyle=':', linewidth=2, label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'{col}', fontweight='bold', color=AMAZON_NAVY)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Numerical Feature Distributions — Histograms with Mean/Median',
             fontsize=13, fontweight='bold', color=AMAZON_NAVY)
plt.tight_layout()
plt.savefig('../reports/figures/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Outlier summary using IQR method
print('Outlier Analysis (IQR Method):')
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f'  {col:<25} Q1={Q1:.1f} | Q3={Q3:.1f} | IQR={IQR:.1f} | Outliers: {outliers} ({outliers/len(df):.1%})')

**Outlier Treatment Decision**: RandomForest is non-parametric and tree splits are ordinal — outlier values within the defined operational range (2–85 km, 15–120 packages, 0–12 days) are valid observations. **No outlier removal applied.**

### 2.3 Relationship Discovery — Correlations

In [ ]:
# Encode categoricals for correlation analysis
df_enc = df.copy()
for col in ['package_type', 'shift', 'carrier', 'weather_risk']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

# Correlation with target
numeric_df = df_enc.drop('package_id', axis=1)
corr_with_target = numeric_df.corr()['delivery_failed'].drop('delivery_failed').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [AMAZON_ORANGE if v > 0.10 else AMAZON_BLUE for v in corr_with_target.values]
bars = ax.barh(corr_with_target.index[::-1], corr_with_target.values[::-1],
                color=colors[::-1], edgecolor='white', height=0.6)
ax.axvline(x=0.10, color=AMAZON_ORANGE, linestyle='--', linewidth=1.5, alpha=0.7, label='Threshold (0.10)')
ax.set_xlabel('|Pearson Correlation| with delivery_failed')
ax.set_title('Feature Correlation with Delivery Failure (Target)',
             fontsize=13, fontweight='bold', color=AMAZON_NAVY, pad=10)
ax.legend()
for bar, val in zip(bars, corr_with_target.values[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr_with_target)

---
## Part 3 — Data Wrangling

### 3.1 Categorical Encoding — LabelEncoder for RandomForest compatibility

In [ ]:
# Encode categorical features (same as train_model.py pipeline)
encoders = {}
encoding_map = {
    'package_type': 'package_type_enc',
    'shift':        'shift_enc',
    'carrier':      'carrier_enc',
    'weather_risk': 'weather_enc'
}

df_wrangled = train.copy()
print('Categorical Encoding Summary')
print('=' * 55)

for src_col, enc_col in encoding_map.items():
    le = LabelEncoder()
    df_wrangled[enc_col] = le.fit_transform(df_wrangled[src_col])
    encoders[src_col] = le
    print(f'  {src_col:<15} → {enc_col:<20}')
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'    Mapping: {mapping}')

print()
print('Note: Encoders fitted on TRAIN only — applied to val/test to prevent leakage')

In [ ]:
# Final feature set for model
MODEL_FEATURES = [
    'package_type_enc', 'shift_enc', 'carrier_enc',
    'route_distance_km', 'packages_in_route',
    'double_scan', 'short_service_time', 'delivery_failed',
    'weather_enc', 'cr_number_missing', 'days_in_fc'
]

X = df_wrangled[MODEL_FEATURES]
y = df_wrangled['delivery_failed']

print('Final Model-Ready Dataset')
print('=' * 40)
print(f'Feature matrix shape : {X.shape}')
print(f'Target shape         : {y.shape}')
print(f'Class balance        : {y.value_counts(normalize=True).to_dict()}')
print()
print('Feature matrix sample (first 3 rows):')
X.head(3)

---
## Part 4 — Final Schema Documentation

In [ ]:
# Final schema table
final_schema = pd.DataFrame([
    {'Column': 'package_id',        'Type': 'string',  'Encoded': '—',                 'Role': 'ID',     'Description': 'Unique package identifier'},
    {'Column': 'package_type',      'Type': 'string',  'Encoded': 'package_type_enc',  'Role': 'Feature','Description': 'Package category (5 types)'},
    {'Column': 'shift',             'Type': 'string',  'Encoded': 'shift_enc',         'Role': 'Feature','Description': 'Delivery shift (morning/afternoon/night)'},
    {'Column': 'carrier',           'Type': 'string',  'Encoded': 'carrier_enc',       'Role': 'Feature','Description': 'Carrier (A=Amazon, B=SEUR, C=DHL, D=Regional Carrier)'},
    {'Column': 'route_distance_km', 'Type': 'float64', 'Encoded': 'as-is',             'Role': 'Feature','Description': 'Route length in km (2–85)'},
    {'Column': 'packages_in_route', 'Type': 'int64',   'Encoded': 'as-is',             'Role': 'Feature','Description': 'Number of packages in driver route (15–120)'},
    {'Column': 'double_scan',       'Type': 'int64',   'Encoded': 'as-is (binary)',    'Role': 'Feature','Description': 'Scan error flag (1=error detected)'},
    {'Column': 'short_service_time',      'Type': 'int64',   'Encoded': 'as-is (binary)',    'Role': 'Feature','Description': 'Planned service time under 25 seconds'},
    {'Column': 'delivery_failed','Type': 'int64',   'Encoded': 'as-is (binary)',    'Role': 'Feature','Description': 'Package damaged at FC receiving'},
    {'Column': 'weather_risk',      'Type': 'string',  'Encoded': 'weather_enc',       'Role': 'Feature','Description': 'Environmental risk level (low/medium/high)'},
    {'Column': 'cr_number_missing', 'Type': 'int64',   'Encoded': 'as-is (binary)',    'Role': 'Feature','Description': 'Customer reference absent from record'},
    {'Column': 'days_in_fc',        'Type': 'int64',   'Encoded': 'as-is',             'Role': 'Feature','Description': 'Days in fulfillment center (0–12)'},
    {'Column': 'delivery_failed',   'Type': 'int64',   'Encoded': 'as-is (binary)',    'Role': 'TARGET', 'Description': 'Delivery outcome: 1=failed, 0=delivered'},
])

print('Final Data Schema')
print(final_schema.to_string(index=False))


## Summary: Data Curation Findings

| Dimension | Result |
|---|---|
| Completeness | ✅ 0% missing values |
| Consistency | ✅ All values within operational ranges |
| Outliers | ✅ No removals needed — RF is robust |
| Encoding | ✅ LabelEncoder applied (train-only fit) |
| Class balance | ⚠️ 80/20 split → addressed with class_weight='balanced' |
| Leakage check | ✅ Encoders fitted on train set only |

**Data is clean, well-structured, and ready for EDA and model training.**

